In [1]:
from IPython.display import clear_output

In [2]:
!pip install stanza
clear_output()

In [3]:
import stanza
stanza.download("hi")        # for German change to "de"
nlp = stanza.Pipeline("hi")

import json
import pandas as pd
import nltk
import re
import time
nltk.download('punkt_tab')
nltk.download('punkt')
from nltk.tokenize import sent_tokenize

import ast
from collections import Counter

from tqdm import tqdm

clear_output()

In [4]:
def extract_verb_triplets(data):
    """
    Extracts subject-verb-object (SVO) triplets from a syntactic dependency-parsed sentence.

    This function expects a syntactically parsed sentence, where `data` is an object that has a `.words` attribute.
    Each element of `data.words` is an object with a `.to_dict()` method, producing a dictionary representing a token
    with at least the following keys:
        - "id": Unique integer identifier for the token
        - "text": The surface text of the token
        - "upos": Universal part-of-speech tag
        - "deprel": Dependency relation to its head
        - "head": ID of the head token

    The function:
        - Identifies all verbs in the sentence.
        - Searches for the corresponding subject and object for each verb.
        - Forms triplets containing a verb, its subject, and its object.

    Returns:
        tuple:
            - An integer count of successfully extracted triplets
            - A list of dictionaries, each representing a triplet with keys: 'verb', 'subject', and 'object'.
              Each key maps to a dictionary containing token information.
            - A list of string abbreviations (e.g., "SVO", "VOS") representing the order of elements in each triplet
              based on their original position in the sentence.
    """
    data = [word.to_dict() for word in data.words]
    triplets = []
    abbr_list = []

    id_to_token = {token["id"]: token for token in data}

    for token in data:
        if token.get("upos") == "VERB":
            verb_id = token["id"]

            subject = next(
                (t for t in data if t.get("head") == verb_id and "subj" in t.get("deprel", "")),
                None
            )

            current_token = token
            while not subject and current_token.get("deprel") == "conj":
                head_id = current_token.get("head")
                head_token = id_to_token.get(head_id)

                if not head_token or head_token.get("upos") != "VERB":
                    break

                candidate_verbs = [
                    t for t in data
                    if t.get("head") == head_token["id"]
                    and t.get("deprel") == "conj"
                    and t.get("id") < current_token["id"]
                    and t.get("upos") == "VERB"
                ]

                for cv in sorted(candidate_verbs, key=lambda x: x["id"]):
                    subj_candidate = next(
                        (t for t in data if t.get("head") == cv["id"] and "subj" in t.get("deprel", "")),
                        None
                    )
                    if subj_candidate:
                        subject = subj_candidate
                        break

                if not subject:
                    subject = next(
                        (t for t in data if t.get("head") == head_token["id"] and "subj" in t.get("deprel", "")),
                        None
                    )

                current_token = head_token

            obj = next(
                (t for t in data if t.get("head") == verb_id and t.get("deprel") == "obj"),
                None
            )

            if subject and obj:
                elements = [
                    {"role": "verb", "data": {k: token[k] for k in ("upos", "text", "id", "deprel", "head")}},
                    {"role": "subject", "data": {k: subject[k] for k in ("upos", "text", "id", "deprel", "head")}},
                    {"role": "object", "data": {k: obj[k] for k in ("upos", "text", "id", "deprel", "head")}},
                ]

                elements_sorted = sorted(elements, key=lambda x: x["data"]["id"])
                triplet = {el["role"]: el["data"] for el in elements_sorted}
                triplets.append(triplet)

                abbr = "".join(el["role"][0].upper() for el in elements_sorted)
                abbr_list.append(abbr)

    return len(triplets), triplets, abbr_list


In [ ]:
with open("/content/hi_qwen_5contexts_top.json", "r", encoding="utf-8") as f:
    data = json.load(f)

texts = [item["gen_answer"] for item in data if "gen_answer" in item]

records = []
text_order_id = -1
for text in tqdm(texts, desc="Processing sentences"):
    text_order_id += 1
    clean_text = re.sub(r'\s+', ' ', text.strip())
    sentences = sent_tokenize(clean_text, language='russian')
    for sentence in sentences:
        doc = nlp(sentence)
        for sent in doc.sentences:
            triplets = extract_verb_triplets(sent)
        records.append({"text_order_id": text_order_id, "sentence": sentence, "num_triplets": triplets[0], "triplets": triplets[1], "abbreviation": triplets[2]})

df = pd.DataFrame(records)
df.to_csv('/content/hi_qwen_5contexts_top.csv', index=False)


In [6]:
df.head()

,text_order_id,sentence,num_triplets,triplets,abbreviation
0,0,झील से उत्तर पूर्व की ओर बढ़ते हुए ग्रूम लेक र...,0,[],[]
1,1,डर्ट-रोड की सुविधा से बड़े खेतों और पशु-फार्मो...,0,[],[]
2,2,ग्रूम झील की सतह ने विमान परीक्षण के लिए एक आद...,1,"[{'subject': {'upos': 'NOUN', 'text': 'सतह', '...",[SOV]
3,3,लॉकहीड विशेषज्ञों की टीम थी।,0,[],[]
4,4,प्रतिबंधित क्षेत्रों में भटकते पर सैन्य पायलटो...,0,[],[]


In [7]:
file_path = "hi_qwen_5contexts_top"
df = pd.read_csv("/content/" + file_path + '.csv')

df["abbreviation"] = df["abbreviation"].apply(ast.literal_eval)

flattened_list = [abbr for sublist in df["abbreviation"] for abbr in sublist]

full_number = len(flattened_list)

empty_lists_count = df["abbreviation"].apply(lambda x: len(x) == 0).sum()

total_number = full_number + empty_lists_count

abbreviation_counts = Counter(flattened_list)

data = []
for abbr, count in abbreviation_counts.items():
    percent_of_full = (count / full_number) * 100
    percent_of_total = (count / total_number) * 100
    data.append({
        "abbreviation": abbr,
        "count": count,
        "percent_of_full": round(percent_of_full, 2), # percentage of the total number of triplets
        "percent_of_total": round(percent_of_total, 2), # percentage of the total number of triplets plus number of sentences without triplets
        "config": file_path
    })

result_df = pd.DataFrame(data)
result_df.to_csv('/content/RESULTS_' + file_path + '.csv', index=False)
result_df

,abbreviation,count,percent_of_full,percent_of_total,config
0,SOV,281,93.98,13.59,hi_qwen_5contexts_top
1,OSV,10,3.34,0.48,hi_qwen_5contexts_top
2,SVO,7,2.34,0.34,hi_qwen_5contexts_top
3,OVS,1,0.33,0.05,hi_qwen_5contexts_top
